In [15]:
!pip install pandas numpy scikit-learn pyspark


   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 3.1 MB/s eta 0:00:03
   -------- ------------------------------- 1.8/8.2 MB 3.7 MB/s eta 0:00:02
   -------------- ------------------------- 2.9/8.2 MB 3.9 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 3.7 MB/s eta 0:00:02
   ---------------------- ----------------- 4.7/8.2 MB 4.0 MB/s eta 0:00:01
   -------------------------- ------------- 5.5/8.2 MB 4.1 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.2 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.2 MB 4.0 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.2 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 3.7 MB/s  0:00:02

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3

In [16]:
import os

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lead

os.environ["HADOOP_HOME"] = r"C:\GraddedAssignment\hadoop"
os.environ["PATH"] += r";C:\GraddedAssignment\hadoop\bin"

spark = (
    SparkSession.builder
    .appName("Telecom ML3 Risk Prediction")
    .master("local[*]")
    .getOrCreate()
)

ML2_PATH = "../dataset/output/ml2_features"
HOURLY_PATH = "../dataset/output/hourly_grid_summary"

RISK_THRESHOLD = 100.0

c:\Users\krishnarajan.r\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [17]:
features_df = spark.read.parquet(ML2_PATH)

features_df = features_df.select(
    "grid_id",
    "feature_timestamp",
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share"
)

print("ML2 features loaded:")
features_df.show(5)

ML2 features loaded:
+-------+-------------------+------------------+--------------------+------------+------------------+-----------------+------------------+
|grid_id|  feature_timestamp|      avg_activity|     activity_growth|active_hours|        peak_ratio|      variability|    internet_share|
+-------+-------------------+------------------+--------------------+------------+------------------+-----------------+------------------+
|     18|2013-11-02 23:00:00|23.584891666666667|-0.02893730042849...|          24| 1.738858951722978|8.336694461753003|0.8826162370189671|
|     18|2013-11-03 00:00:00|23.604541666666663|-0.01870986561129...|          24|1.7374114091744355|8.318457721232196|0.8827464347433138|
|     18|2013-11-03 01:00:00| 23.47452083333333|-0.01862730086863...|          24|1.7470345951328439|8.435636456644959|0.8817254878294463|
|     18|2013-11-03 02:00:00|23.558441666666667|-0.00207808253026...|          24|1.7408112378683793|8.346299386992946|0.8837545140400839|
|     

In [18]:
hourly_df = spark.read.parquet(HOURLY_PATH)

hourly_df = hourly_df.select(
    "grid_id",
    "timestamp",
    "total_activity"
)

hourly_df = hourly_df.withColumn(
    "timestamp",
    col("timestamp").cast("timestamp")
)

print("Hourly activity loaded:")
hourly_df.show(5)

Hourly activity loaded:
+-------+-------------------+------------------+
|grid_id|          timestamp|    total_activity|
+-------+-------------------+------------------+
|     14|2013-11-06 00:00:00|            11.706|
|     55|2013-11-06 00:00:00|            78.502|
|     97|2013-11-06 00:00:00|50.615199999999994|
|    132|2013-11-06 00:00:00|           41.2393|
|    157|2013-11-06 00:00:00|           54.0591|
+-------+-------------------+------------------+
only showing top 5 rows


In [19]:
from pyspark.sql.window import Window

future_window = (
    Window
    .partitionBy("grid_id")
    .orderBy("timestamp")
)

hourly_df = hourly_df.withColumn(
    "future_activity",
    lead("total_activity", 1).over(future_window)
)

hourly_df = hourly_df.select(
    "grid_id",
    "timestamp",
    "future_activity"
)

hourly_df.show(10)

+-------+-------------------+------------------+
|grid_id|          timestamp|   future_activity|
+-------+-------------------+------------------+
|     31|2013-11-01 00:00:00|26.656300000000005|
|     31|2013-11-01 01:00:00|           25.7551|
|     31|2013-11-01 02:00:00|           27.5047|
|     31|2013-11-01 03:00:00|           26.2878|
|     31|2013-11-01 04:00:00|           26.8677|
|     31|2013-11-01 05:00:00|           29.9044|
|     31|2013-11-01 06:00:00|           30.5244|
|     31|2013-11-01 07:00:00|42.145799999999994|
|     31|2013-11-01 08:00:00|            56.785|
|     31|2013-11-01 09:00:00|           65.1998|
+-------+-------------------+------------------+
only showing top 10 rows


In [20]:
ml3_spark_df = features_df.join(
    hourly_df,
    (features_df.grid_id == hourly_df.grid_id) &
    (features_df.feature_timestamp == hourly_df.timestamp),
    "inner"
).select(
    features_df.grid_id,
    features_df.feature_timestamp,
    features_df.avg_activity,
    features_df.activity_growth,
    features_df.active_hours,
    features_df.peak_ratio,
    features_df.variability,
    features_df.internet_share,
    hourly_df.future_activity
)

print("ML3 dataset:")
ml3_spark_df.show(10)

ML3 dataset:
+-------+-------------------+-----------------+--------------------+------------+------------------+------------------+------------------+------------------+
|grid_id|  feature_timestamp|     avg_activity|     activity_growth|active_hours|        peak_ratio|       variability|    internet_share|   future_activity|
+-------+-------------------+-----------------+--------------------+------------+------------------+------------------+------------------+------------------+
|      1|2013-11-02 23:00:00|72.80891250000002|0.015349765281404108|          24|1.6798479169703293| 27.00553802621768|0.8673626208421482|           57.4846|
|      1|2013-11-03 01:00:00|73.21688333333334|   0.028471100176371|          24|1.6704876584703936|26.675385890981108|0.8682140253170203|41.650600000000004|
|      1|2013-11-03 06:00:00|       73.3472875|0.034718791368019225|          24|1.6675176979107784|26.515830471860497|0.8684206056472552|           39.1188|
|      1|2013-11-03 10:00:00|72.9260083

In [21]:
ml3_spark_df = ml3_spark_df.filter(
    col("future_activity").isNotNull()
)

ml3_spark_df = ml3_spark_df.withColumn(
    "target",
    (col("future_activity") > RISK_THRESHOLD).cast("int")
)

ml3_spark_df.show(10)

+-------+-------------------+-----------------+--------------------+------------+------------------+------------------+------------------+------------------+------+
|grid_id|  feature_timestamp|     avg_activity|     activity_growth|active_hours|        peak_ratio|       variability|    internet_share|   future_activity|target|
+-------+-------------------+-----------------+--------------------+------------+------------------+------------------+------------------+------------------+------+
|      1|2013-11-02 23:00:00|72.80891250000002|0.015349765281404108|          24|1.6798479169703293| 27.00553802621768|0.8673626208421482|           57.4846|     0|
|      1|2013-11-03 01:00:00|73.21688333333334|   0.028471100176371|          24|1.6704876584703936|26.675385890981108|0.8682140253170203|41.650600000000004|     0|
|      1|2013-11-03 06:00:00|       73.3472875|0.034718791368019225|          24|1.6675176979107784|26.515830471860497|0.8684206056472552|           39.1188|     0|
|      1|2

In [22]:
ml3_spark_df = ml3_spark_df.filter(
    col("future_activity").isNotNull()
)

ml3_spark_df = ml3_spark_df.withColumn(
    "target",
    (col("future_activity") > RISK_THRESHOLD).cast("int")
)

ml3_spark_df.show(10)

+-------+-------------------+-----------------+--------------------+------------+------------------+------------------+------------------+------------------+------+
|grid_id|  feature_timestamp|     avg_activity|     activity_growth|active_hours|        peak_ratio|       variability|    internet_share|   future_activity|target|
+-------+-------------------+-----------------+--------------------+------------+------------------+------------------+------------------+------------------+------+
|      1|2013-11-02 23:00:00|72.80891250000002|0.015349765281404108|          24|1.6798479169703293| 27.00553802621768|0.8673626208421482|           57.4846|     0|
|      1|2013-11-03 01:00:00|73.21688333333334|   0.028471100176371|          24|1.6704876584703936|26.675385890981108|0.8682140253170203|41.650600000000004|     0|
|      1|2013-11-03 06:00:00|       73.3472875|0.034718791368019225|          24|1.6675176979107784|26.515830471860497|0.8684206056472552|           39.1188|     0|
|      1|2

In [23]:
ml3_df = ml3_spark_df.toPandas()

ml3_df = ml3_df.sort_values(
    "feature_timestamp"
).reset_index(drop=True)

print("ML3 rows:", len(ml3_df))

print("\nClass balance:")
print(ml3_df["target"].value_counts())

print("\nClass proportion:")
print(ml3_df["target"].value_counts(normalize=True))

c:\Users\krishnarajan.r\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\krishnarajan.r\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


ML3 rows: 1199994

Class balance:
target
1    860632
0    339362
Name: count, dtype: int64

Class proportion:
target
1    0.717197
0    0.282803
Name: proportion, dtype: float64


In [24]:
split_index = int(len(ml3_df) * 0.8)

train_df = ml3_df.iloc[:split_index].copy()
test_df = ml3_df.iloc[split_index:].copy()

print("TRAIN")
print("Earliest:", train_df["feature_timestamp"].min())
print("Latest  :", train_df["feature_timestamp"].max())

print("\nTEST")
print("Earliest:", test_df["feature_timestamp"].min())
print("Latest  :", test_df["feature_timestamp"].max())

TRAIN
Earliest: 2013-11-02 23:00:00
Latest  : 2013-11-06 22:00:00

TEST
Earliest: 2013-11-06 23:00:00
Latest  : 2013-11-07 22:00:00


In [25]:
FEATURES = [
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share"
]

X_train = train_df[FEATURES]
y_train = train_df["target"]

X_test = test_df[FEATURES]
y_test = test_df["target"]

print("Features:")
print(FEATURES)

print("\nTraining rows:", len(X_train))
print("Testing rows :", len(X_test))

Features:
['avg_activity', 'activity_growth', 'active_hours', 'peak_ratio', 'variability', 'internet_share']

Training rows: 959995
Testing rows : 239999


In [26]:
model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train,
    y_train
)

print("Logistic Regression training completed")

Logistic Regression training completed


In [27]:
y_pred = model.predict(X_test)

test_df["prediction"] = y_pred

print("Predictions generated")

test_df[
    [
        "grid_id",
        "feature_timestamp",
        "future_activity",
        "target",
        "prediction"
    ]
].head(20)

Predictions generated


,grid_id,feature_timestamp,future_activity,target,prediction
959995,8959,2013-11-06 23:00:00,736.8699,1,1
959996,3820,2013-11-06 23:00:00,48.2481,0,0
959997,2193,2013-11-06 23:00:00,174.3357,1,1
959998,3060,2013-11-06 23:00:00,61.0959,0,1
959999,5632,2013-11-06 23:00:00,51.0243,0,0
960000,4052,2013-11-06 23:00:00,549.8737,1,1
960001,4897,2013-11-06 23:00:00,31.3811,0,0
960002,7198,2013-11-06 23:00:00,118.7763,1,1
960003,8840,2013-11-06 23:00:00,410.1127,1,1
960004,6929,2013-11-06 23:00:00,66.6101,0,1


In [28]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

base_rate = y_test.mean()

print("ML3 Evaluation")
print("-----------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("Base rate:", base_rate)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

ML3 Evaluation
-----------------------
Accuracy : 0.8876161984008266
Precision: 0.9287094371246334
Recall   : 0.9155383697448136
Base rate: 0.7262696927903867

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.81      0.80     65695
           1       0.93      0.92      0.92    174304

    accuracy                           0.89    239999
   macro avg       0.86      0.86      0.86    239999
weighted avg       0.89      0.89      0.89    239999


Confusion Matrix:
[[ 53445  12250]
 [ 14722 159582]]


In [29]:
coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print(coefficients)

           feature  coefficient  absolute_coefficient
3       peak_ratio     0.989793              0.989793
5   internet_share    -0.401680              0.401680
2     active_hours    -0.195457              0.195457
1  activity_growth     0.069177              0.069177
0     avg_activity     0.042293              0.042293
4      variability    -0.035165              0.035165


In [30]:
prediction_results = test_df[
    [
        "grid_id",
        "feature_timestamp",
        "future_activity",
        "target",
        "prediction"
    ]
].copy()

prediction_results["prediction_label"] = np.where(
    prediction_results["prediction"] == 1,
    "RISK",
    "NO RISK"
)

prediction_results.head(20)

,grid_id,feature_timestamp,future_activity,target,prediction,prediction_label
959995,8959,2013-11-06 23:00:00,736.8699,1,1,RISK
959996,3820,2013-11-06 23:00:00,48.2481,0,0,NO RISK
959997,2193,2013-11-06 23:00:00,174.3357,1,1,RISK
959998,3060,2013-11-06 23:00:00,61.0959,0,1,RISK
959999,5632,2013-11-06 23:00:00,51.0243,0,0,NO RISK
960000,4052,2013-11-06 23:00:00,549.8737,1,1,RISK
960001,4897,2013-11-06 23:00:00,31.3811,0,0,NO RISK
960002,7198,2013-11-06 23:00:00,118.7763,1,1,RISK
960003,8840,2013-11-06 23:00:00,410.1127,1,1,RISK
960004,6929,2013-11-06 23:00:00,66.6101,0,1,RISK


decision tree



In [31]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(
    X_train,
    y_train
)

print("Decision Tree training completed")

Decision Tree training completed


In [32]:
tree_pred = tree_model.predict(X_test)

print("Decision Tree predictions generated")

tree_accuracy = accuracy_score(y_test, tree_pred)

tree_precision = precision_score(
    y_test,
    tree_pred,
    zero_division=0
)

tree_recall = recall_score(
    y_test,
    tree_pred,
    zero_division=0
)

print("Decision Tree Evaluation")
print("-----------------------")
print("Accuracy :", tree_accuracy)
print("Precision:", tree_precision)
print("Recall   :", tree_recall)
print("Base rate:", y_test.mean())

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        tree_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, tree_pred))

Decision Tree predictions generated
Decision Tree Evaluation
-----------------------
Accuracy : 0.898978745744774
Precision: 0.9040643663799836
Recall   : 0.9631046906554066
Base rate: 0.7262696927903867

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.73      0.80     65695
           1       0.90      0.96      0.93    174304

    accuracy                           0.90    239999
   macro avg       0.89      0.85      0.87    239999
weighted avg       0.90      0.90      0.90    239999


Confusion Matrix:
[[ 47881  17814]
 [  6431 167873]]


In [34]:
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall"
    ],
    "Logistic Regression": [
        accuracy,
        precision,
        recall
    ],
    "Decision Tree": [
        tree_accuracy,
        tree_precision,
        tree_recall
    ]
})

comparison

,Metric,Logistic Regression,Decision Tree
0,Accuracy,0.887616,0.898979
1,Precision,0.928709,0.904064
2,Recall,0.915538,0.963105


Decision Tree was selected as the preferred model because it achieved higher test accuracy (89.90%) and substantially higher recall (96.31%) than Logistic Regression (88.76% accuracy and 91.55% recall). Although Logistic Regression achieved higher precision, the Decision Tree is preferable for the operational objective of identifying cases that warrant investigation, where missing a potentially risky interval is more costly than generating some additional alerts.


In [37]:
import os
import joblib

MODEL_PATH = r"C:\GraddedAssignment\dataset\output\ml5_model"

os.makedirs(MODEL_PATH, exist_ok=True)

model_file = os.path.join(
    MODEL_PATH,
    "decision_tree.joblib"
)

joblib.dump(
    tree_model,
    model_file
)

print("Model saved:")
print(model_file)

print("Exists:", os.path.exists(model_file))

Model saved:
C:\GraddedAssignment\dataset\output\ml5_model\decision_tree.joblib
Exists: True


In [38]:
import os
import joblib

MODEL_PATH = r"C:\GraddedAssignment\dataset\output\ml5_model"
model_file = os.path.join(MODEL_PATH, "decision_tree.joblib")

print("Model variable type:")
print(type(tree_model))

print("\nFolder:")
print(MODEL_PATH)

print("\nFolder exists before save:")
print(os.path.exists(MODEL_PATH))

os.makedirs(MODEL_PATH, exist_ok=True)

print("\nSaving model...")

result = joblib.dump(tree_model, model_file)

print("\njoblib returned:")
print(result)

print("\nFile path:")
print(model_file)

print("\nFile exists after save:")
print(os.path.exists(model_file))

print("\nFolder contents:")
print(os.listdir(MODEL_PATH))

Model variable type:
<class 'sklearn.tree._classes.DecisionTreeClassifier'>

Folder:
C:\GraddedAssignment\dataset\output\ml5_model

Folder exists before save:
True

Saving model...

joblib returned:
['C:\\GraddedAssignment\\dataset\\output\\ml5_model\\decision_tree.joblib']

File path:
C:\GraddedAssignment\dataset\output\ml5_model\decision_tree.joblib

File exists after save:
True

Folder contents:
['decision_tree.joblib']
